# Protocol A, quantity sweep: imports and config hash

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

try:
    import core
except ModuleNotFoundError:
    hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
    if not hits:
        raise ModuleNotFoundError("core.py not found.")
    sys.path.insert(0, os.path.dirname(hits[0]))
    import core

print("import-time CONFIG_HASH:", core.CONFIG_HASH)

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1


# GPU guard

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device. Protocol A trains 63 runs including ResNet-18; "
        "a CPU kernel is infeasible. Switch the Kaggle accelerator to GPU (P100) and rerun. "
        "Notebook 00 can run on CPU, notebook 01 cannot."
    )
print("CUDA available:", torch.cuda.get_device_name(0))

CUDA available: Tesla T4


# Stage the frozen config, refresh, then load data

In [4]:
if core.DATAWISE_RESULTS_SLUG is not None:
    try:
        core.pull_results_store()
        print("pulled results store from", core.DATAWISE_RESULTS_SLUG)
    except Exception as exc:
        print("pull skipped (first session or empty store):", exc)

frozen_target = core.PATHS["frozen_config"]
if not os.path.exists(frozen_target):
    hits = glob.glob("/kaggle/input/**/frozen_config.md", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "frozen_config.md not found in the working dir or any mounted dataset. "
            "Run 00_setup.ipynb and mount the results store (or the dataset holding "
            "frozen_config.md) before running this notebook."
        )
    core._ensure(os.path.dirname(frozen_target))
    shutil.copy(hits[0], frozen_target)
    print("staged frozen_config.md from", hits[0])

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)
print("core.NORM_MEAN now:", core.NORM_MEAN)
print("core.NORM_STD  now:", core.NORM_STD)

data = core.load_pcam()

Dataset URL: https://www.kaggle.com/datasets/hrushisanap/datawise-results
pulled results store from hrushisanap/datawise-results
active CONFIG_HASH after refresh: 03fa342b535795c1
core.NORM_MEAN now: [0.700756, 0.538358, 0.691621]
core.NORM_STD  now: [0.234978, 0.277409, 0.212895]
[11:18:00] train pos/neg 131072/131072
[11:18:00] valid pos/neg (raw) 16369/16399
[11:18:00] test pos/neg (raw) 16377/16391
[11:18:01] valid balanced 32738 (16369/16369)
[11:18:01] test balanced 32754 (16377/16377)


# Build the Protocol A condition list

In [5]:
conditions = [(s, s) for s in core.PROTOCOL_A_SIZES]

n_pos = len(data.train_pos_idx)
n_neg = len(data.train_neg_idx)

print(f"{'idx':>3}  {'n_min':>7}  {'n_maj':>7}  {'total N':>8}")
for i, (n_min, n_maj) in enumerate(conditions):
    assert n_min <= n_pos, f"n_min {n_min} exceeds positive pool {n_pos}"
    assert n_maj <= n_neg, f"n_maj {n_maj} exceeds negative pool {n_neg}"
    print(f"{i:>3}  {n_min:>7}  {n_maj:>7}  {n_min + n_maj:>8}")

print(f"\n{len(conditions)} conditions, {len(core.MODEL_NAMES)} models, "
      f"{len(core.SEEDS)} seeds -> "
      f"{len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)} run_training calls")

idx    n_min    n_maj   total N
  0     2000     2000      4000
  1     4000     4000      8000
  2     8000     8000     16000
  3    16000    16000     32000
  4    32000    32000     64000
  5    64000    64000    128000
  6   128000   128000    256000

7 conditions, 3 models, 3 seeds -> 63 run_training calls


# Run the sweep (models outermost, then conditions, then seeds)

In [6]:
done = 0
for model in core.MODEL_NAMES:
    for (n_min, n_maj) in conditions:
        for s in core.SEEDS:
            run_id = core.run_training(
                "protocolA", model, n_min, n_maj, s, s, s, data,
                target_ratio="", resume=True)
            done += 1
            print(f"[{done:>2}/63] {model} m={n_min} M={n_maj} seed={s} -> {run_id}")

print("sweep calls completed:", done)

[11:18:01] logistic parameter count: 55298
[11:18:12] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 250/5000 loss=0.5356
[11:18:22] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 500/5000 loss=0.6793
[11:18:32] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 750/5000 loss=0.5518
[11:18:42] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 1000/5000 loss=0.5313
[11:18:52] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 1250/5000 loss=0.6173
[11:19:02] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 1500/5000 loss=0.6713
[11:19:12] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 1750/5000 loss=0.3806
[11:19:22] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 2000/5000 loss=0.4381
[11:19:32] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 2250/5000 loss=0.6000
[11:19:42] [protocolA][logistic][m=2000 M=2000 r=1.00][seed=21] step 2500/5000 loss=0.5712
[11:19:51] [protocolA][logistic][m=2000 M=2000 r=1

# Session summary from the ledger

In [7]:
if not os.path.exists(core.PATHS["ledger"]):
    raise FileNotFoundError("no ledger yet; run the sweep cell first")

ledger = pd.read_csv(core.PATHS["ledger"])
mask = ledger["protocol"].astype(str).str.contains("protocolA")
rows = ledger[mask]

order = {"failed": 0, "success": 1}
rows = rows.assign(_rank=rows["status"].map(order).fillna(-1))
dedup = rows.sort_values("_rank").drop_duplicates("run_id", keep="last").drop(columns="_rank")
n_success = int((dedup["status"] == "success").sum())
n_failed = int((dedup["status"] == "failed").sum())
expected = len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)

print(f"Protocol A: {n_success} success, {n_failed} failed, "
      f"{expected - n_success} pending of {expected}")

failed_rows = dedup[dedup["status"] == "failed"]
if len(failed_rows):
    print("\nfailed runs:")
    for _, r in failed_rows.iterrows():
        print(f"  {r['run_id']}  (model={r['model']} "
              f"n_min={r['n_min']} n_maj={r['n_maj']} seed={r['data_seed']})")

core._ensure(core.PATHS["summaries"])
summary_path = os.path.join(core.PATHS["summaries"], "01_protocolA.md")
lines = [
    "# Notebook 01: Protocol A (quantity) session summary",
    "",
    f"Active config hash: {core.CONFIG_HASH}",
    "",
    f"- expected unique runs: {expected} (7 conditions x 3 models x 3 seeds)",
    f"- success: {n_success}",
    f"- failed: {n_failed}",
    f"- pending: {expected - n_success}",
    "",
    "## Failed runs",
]
if len(failed_rows):
    for _, r in failed_rows.iterrows():
        lines.append(f"- {r['run_id']} (model={r['model']} "
                     f"n_min={r['n_min']} n_maj={r['n_maj']} seed={r['data_seed']})")
else:
    lines.append("- none")
lines.append("")
with open(summary_path, "w") as f:
    f.write("\n".join(lines))
print("\nwrote", summary_path)

Protocol A: 63 success, 0 failed, 0 pending of 63

wrote /kaggle/working/summaries/01_protocolA.md


# Persistence

In [8]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session.")
    print("WARNING: without it, the ledger and logits will not survive session end.")

Starting upload for file subsets.zip


100%|██████████| 4.36M/4.36M [00:00<00:00, 8.94MB/s]


Upload successful: subsets.zip (4MB)
Starting upload for file __notebook__.ipynb


100%|██████████| 267k/267k [00:00<00:00, 680kB/s]


Upload successful: __notebook__.ipynb (267KB)
Starting upload for file logits.zip


...

Upload successful: frozen_config.md (431B)
Starting upload for file balanced_eval.zip


100%|██████████| 139k/139k [00:00<00:00, 357kB/s]


Upload successful: balanced_eval.zip (139KB)
Starting upload for file summaries.zip


100%|██████████| 819/819 [00:00<00:00, 2.06kB/s]


Upload successful: summaries.zip (819B)
pushed results store to hrushisanap/datawise-results
